# Phase 8 — Risque

## Pourquoi cette phase est importante

La phase 6 mesure la concentration des poids : dix premières lignes, nombre effectif de titres, multiple. La phase 5 mesure une volatilité et un drawdown maximal. Cette phase décrit la distribution des rendements mensuels : dans quelle mesure les titres évoluent ensemble, et quelle perte mensuelle la série a connue dans sa queue.

Elle ne refait pas la concentration des poids. Un lecteur qui cherche le top 10 reste sur la phase 6.

## Où en est le calcul

La méthode est fixée ici. Le module n'est pas encore dans `src/`. La volatilité et le drawdown maximal déjà publiés sont ceux de la phase 5.


## Sources lues

Deux familles de fichiers. Le risque du portefeuille se lit sur son rendement mensuel. La corrélation se lit sur les titres, donc il faut le panneau de cours déjà converti.

| Fichier | Origine | Colonnes | Usage |
|---|---|---|---|
| `data/04_portfolios/portfolio_monthly_returns.csv` | phase 5 | `month`, `strategy`, `portfolio_return` | Volatilité, VaR, expected shortfall |
| `data/04_portfolios/portfolio_membership.csv` | phase 5 | `yahoo_symbol`, `decision` | Les titres `included`, les seuls du panneau |
| `data/02_processed/market/market_prices_chf.csv` | `DS-003` et `DS-004` | `symbol`, `timestamp`, `close_chf` | Rendements mensuels des titres, pour la corrélation |
| `data/02_processed/macro/saron.csv` | `DS-005` | `date` | La même période que la phase 5. Le risque est mesuré sur ces mois, pas sur un autre calendrier |

`DS-006` n'est pas lu. Un statut ODD Indeterminate ne change pas la queue de la distribution.

`DS-000` n'est pas lu non plus. Le secteur et la zone sont le sujet de la phase 6.


## Volatilité

Même définition que la phase 5, rappelée pour que la page se lise seule. $s$ est l'écart-type d'échantillon des rendements du portefeuille, diviseur $T - 1$.

$$
\sigma_p^{\mathrm{ann}} = s(R_p) \times \sqrt{12}
$$

Le nombre doit égaler `annualized_volatility` de `portfolio_metrics.csv`. Sinon le calcul s'arrête.


## Corrélation

On prend les titres dont `decision` vaut `included`. Leur rendement mensuel est celui de la phase 5 : dernier cours CHF du mois, divisé par le dernier cours CHF du mois précédent, moins un. La période est celle du SARON.

La corrélation publiée entre deux titres est la corrélation de Pearson de ces deux séries, sur les $T$ mois communs. Elle vaut 1 pour un titre avec lui-même.

Le résumé du portefeuille n'est pas la matrice entière. C'est la moyenne des corrélations hors diagonale, une seule fois par paire :

$$
\bar\rho = \frac{2}{N(N-1)} \sum_{i < j} \rho_{ij}
$$

$N$ est le nombre de titres inclus. Cette moyenne décrit l'univers partagé. Elle est donc la même pour les trois stratégies. Elle dit si la liste bouge ensemble. Elle ne dit pas quel poids chaque stratégie a choisi : ce rôle reste à la phase 6.

Une paire dont une série serait constante n'arrive pas jusqu'ici. La phase 5 a déjà écarté ces titres.


## Value at Risk historique

La VaR retenue est historique. On ne suppose pas que les rendements suivent une loi normale. Avec une période d'une centaine de mois, la queue est courte : le chiffre changera lorsque la période changera. C'est accepté.

On trie les $T$ rendements mensuels du portefeuille du plus mauvais au meilleur. Le niveau retenu est 95 %. Le rang, en comptant à partir de 1, est

$$
k = \lceil 0{,}05 \times T \rceil
$$

La VaR 95 % est le rendement qui occupe ce rang. Elle est en général négative. Une VaR de $-0{,}08$ se lit : dans les 5 % de mois les plus mauvais de cette période, le pire seuil atteint est une perte de 8 % sur le mois. Ce n'est pas une prévision du mois suivant. C'est un résumé de la série déjà observée.

L'expected shortfall 95 % est la moyenne des rendements de rang 1 à $k$, donc la moyenne des mois au moins aussi mauvais que la VaR. Il est toujours inférieur ou égal à la VaR, puisque la VaR est le moins mauvais de ce groupe.

Les deux nombres sont calculés séparément pour chaque stratégie, sur `portfolio_return`.


## Fichiers prévus

`data/07_risk/risk_by_strategy.csv`, une ligne par stratégie : volatilité annualisée, VaR 95 %, expected shortfall 95 %.

`data/07_risk/universe_correlation.csv`, une ligne : nombre de titres inclus, corrélation moyenne hors diagonale, nombre de mois.

La cellule suivante vérifie la présence des fichiers d'entrée. Elle ne calcule pas la VaR.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
files = [
    "data/04_portfolios/portfolio_monthly_returns.csv",
    "data/04_portfolios/portfolio_membership.csv",
    "data/02_processed/market/market_prices_chf.csv",
    "data/02_processed/macro/saron.csv"
]
print("Fichiers lus par cette phase")
for relative in files:
    path = PROJECT_ROOT / relative
    state = "présent" if path.exists() else "absent"
    print(f"{state}  {relative}")
print("Le module de risque n'est pas encore dans src/.")


Fichiers lus par cette phase
présent  data/04_portfolios/portfolio_monthly_returns.csv
présent  data/04_portfolios/portfolio_membership.csv
présent  data/02_processed/market/market_prices_chf.csv
présent  data/02_processed/macro/saron.csv
Le module de risque n'est pas encore dans src/.


# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas. Les formules restent dans la section qui les écrit.

**Corrélation de Pearson.** Lien entre les rendements mensuels de deux titres sur les mois communs. Elle vaut 1 pour un titre avec lui-même.

**Corrélation moyenne.** Moyenne des corrélations hors diagonale, une fois par paire. Elle décrit la liste partagée, pas les poids d'une stratégie.

**Drawdown maximal.** Plus forte baisse déjà publiée par la phase 5. Cette note ne la recalcule pas.

**Expected shortfall.** Moyenne des rendements mensuels au moins aussi mauvais que la VaR du même seuil.

**Indeterminate.** Statut ODD non lu. Il ne change pas la queue de la distribution.

**Queue.** Partie extrême de la série des rendements, celle des mois les plus mauvais.

**SARON.** Fichier relu pour garder la même période que la phase 5.

**Value at Risk historique (VaR).** Rendement du rang qui correspond aux 5 % de mois les plus mauvais, sans supposer une loi normale. Une VaR de −0,08 se lit comme une perte de 8 % sur le mois dans cette queue observée. Ce n'est pas une prévision.

**Volatilité.** Écart-type d'échantillon des rendements du portefeuille, annualisé avec la racine de 12. Même définition que la phase 5.
